Currency Conversion Tool

In [ ]:
import os
os.environ['OPENAI_API_KEY']=''
os.environ['OPENAI_API_VERSION']=''
os.environ['OPENAI_AZURE_ENDPOINT']=''
os.environ['OPENAI_AZURE_MODEL']=''

In [ ]:
from langchain_openai import AzureChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [ ]:
# create tool
from langchain_core.tools import InjectedToolArg
from typing import Annotated
exchange_rate_api_key=''
@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """
    This function fetches the currency conversion factor between a given base and a target currency
    """
    url=f'https://v6.exchangerate-api.com/v6/{exchange_rate_api_key}/pair/{base_currency}/{target_currency}'
    response=requests.get(url)
    return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
    """
    Given a currency conversion rate this function calculates the target currency value from a given base currency value
    """
    return base_currency_value * conversion_rate

In [ ]:
get_conversion_factor.invoke({'base_currency': 'USD', 'target_currency':'INR'})

In [ ]:
convert.invoke({'base_currency_value': 11, 'conversion_rate': 88.24})

In [ ]:
# tool binding

llm=AzureChatOpenAI(
        api_key=os.environ['OPENAI_API_KEY'],
        api_version=os.environ['OPENAI_API_VERSION'],
        azure_endpoint=os.environ['OPENAI_AZURE_ENDPOINT'],
        model_name=os.environ['OPENAI_AZURE_MODEL'],
        temperature=0.4,
        max_tokens=1000,
        seed=42,
)

In [ ]:
llm_with_tool=llm.bind_tools([get_conversion_factor, convert])
llm_with_tool

In [ ]:
# tool calling
messages=[HumanMessage('What is the conversion factor between USD and INR, and based on that can you convert 15 usd to inr')]
messages

In [ ]:
ai_message=llm_with_tool.invoke(messages)
messages.append(ai_message)
ai_message.tool_calls

In [ ]:
import json
for tool_call in ai_message.tool_calls:
    # execute the 1st tool and get the value of conversion rate
    if tool_call['name']=='get_conversion_factor':
        tool_message1=get_conversion_factor.invoke(tool_call)
        print(tool_message1)
        # fetch this conversion rate
        conversion_rate=json.loads(tool_message1.content)['conversion_rate']
        # append this tool message to message list
        messages.append(tool_message1)
    # execute the 2nd tool using the conversion rate from tool 1
    if tool_call['name'] == 'convert':
        # fetch the current arg
        tool_call['args']['conversion_rate']=conversion_rate    ### Adds a new argument called "conversion_rate" to the tool’s arguments.
        tool_message2=convert.invoke(tool_call)
        messages.append(tool_message2)

In [ ]:
messages

In [ ]:
llm_with_tool.invoke(messages).content